# Data Preprocessing and Vector Database Initialisation

This notebook handles the construction of the Evaluation Vector Database ($V_{eval}$) defined in Section 4.2 of the thesis.

## Objective
The script connects to the local `MongoDB` instance containing the raw Guardian news articles and processes them into semantically embedded chunks stored in a local `ChromaDB` instance. The embedding model used is Google's `gemini-embedding-001`.

## Methodological Alignment
To ensure compatibility with the *Mulit-Hop Construction and Expansion* mechanism (Section 4.3), the pipeline enforces the following structural requirements:
* **Chunking Strategy:** Articles are processed using recursive character splitting (size: 1000, overlap: 150) to preserve context continuity.
* **Context Augmentation:** Each chunk is prepended with its source headline and publication date to improve the LLM's grounding during generation.
* **Structural Metadata Tracking:** Each chunk's metadata is enriched with specific identifiers: the `article_id` (representing the parent document $P(d_i)$), the `chunk_index` (its sequential position $Idx(d_i)$), and the `total_chunks` (the document's total size $N(d_i)$). The parent-document identifier is crucial for the framework's multi-hop progression, ensuring that generated thematic bridges span strictly across disjoint documents ($P(d_A) \neq P(d_B)$). Furthermore, the positional metadata enables the deterministic context expansion mechanism during evolving multi-turn interactions.

## Prerequisites
* A running MongoDB instance containing the `guardian_db` database (populated via the fetcher script).
* A `.env` file containing the `MONGO_URI` and the Google Gemini API key.

## References:

* Guardian-API: https://open-platform.theguardian.com/documentation/
* MongoDB Docker: https://hub.docker.com/_/mongo

In [3]:
import os
import chromadb
from pymongo import MongoClient
from google import genai
from google.genai import types
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import time

In [4]:
load_dotenv(override=True)
client = genai.Client()

mongo_client = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017/'))
db = mongo_client.guardian_db

chroma_client = chromadb.PersistentClient(path="./chroma_db")
vector_collection = chroma_client.get_or_create_collection(name="guardian_chunks")

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=True
)
def get_embeddings_with_retry(texts):
    SUB_BATCH_SIZE = 100 
    all_embeddings = []

    for i in range(0, len(texts), SUB_BATCH_SIZE):
        batch = texts[i:i + SUB_BATCH_SIZE]
        while True:
            try:
                response = client.models.embed_content(
                    model='gemini-embedding-001',
                    contents=batch,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                all_embeddings.extend([emb.values for emb in response.embeddings])
                break
            except Exception as e:
                if "429" in str(e):
                    time.sleep(5) 
                else:
                    raise e
    return all_embeddings

def process_batch(articles):
    all_chunk_texts = []
    all_chunk_ids = []
    all_chunk_metas = []
    processed_article_ids = []

    for doc in articles:
        article_id = doc['id']
        headline = doc.get('fields', {}).get('headline', doc.get('webTitle', ''))
        date = doc.get('webPublicationDate', '')[:10]
        body = doc.get('fields', {}).get('bodyText', '')

        if not body or len(body.strip()) < 10:
            db.articles.update_one({"id": article_id}, {"$set": {"embedded": True, "error": "Empty body"}})
            continue

        raw_chunks = text_splitter.split_text(body)
        total_chunks = len(raw_chunks)
        for i, c in enumerate(raw_chunks):
            all_chunk_texts.append(f"Title: {headline}\nDate: {date}\n\nContent: {c}")
            all_chunk_ids.append(f"{article_id}#chunk_{i}")
            all_chunk_metas.append({
                "article_id": article_id,
                "chunk_index": i,
                "date": date,
                "total_chunks": total_chunks
            })
        
        processed_article_ids.append(article_id)

    if not all_chunk_texts:
        return

    try:
        embeddings = get_embeddings_with_retry(all_chunk_texts)

        vector_collection.add(
            ids=all_chunk_ids,
            embeddings=embeddings,
            documents=all_chunk_texts,
            metadatas=all_chunk_metas
        )

        db.articles.update_many({"id": {"$in": processed_article_ids}}, {"$set": {"embedded": True}})
        
        time.sleep(0.1) 

    except Exception as e:
        print(f"Batch-Fehler: {e}")

def process_everything():
    total_remaining = db.articles.count_documents({"embedded": {"$ne": True}})
    print(f"Turbo-Start: {total_remaining} Artikel verbleibend.")
    pbar = tqdm(total=total_remaining, desc="Ingestion")

    while True:
        # Größere Batches aus der DB holen (50-100 Artikel auf einmal)
        articles = list(db.articles.find({"embedded": {"$ne": True}}).limit(50))
        if not articles: break
        process_batch(articles)
        pbar.update(len(articles))
    
    pbar.close()

In [40]:
process_everything()

Turbo-Start: 21152 Artikel verbleibend.





Ingestion:   0%|                                      | 0/21152 [00:00<?, ?it/s]


Ingestion:   0%|                           | 50/21152 [00:08<1:00:05,  5.85it/s]


Ingestion:   0%|                          | 100/21152 [00:18<1:07:01,  5.23it/s]


Ingestion:   1%|▏                         | 150/21152 [00:29<1:09:43,  5.02it/s]


Ingestion:   1%|▏                         | 200/21152 [00:38<1:08:55,  5.07it/s]


Ingestion:   1%|▎                         | 250/21152 [00:49<1:11:23,  4.88it/s]


Ingestion:   1%|▎                         | 300/21152 [01:07<1:28:47,  3.91it/s]


Ingestion:   2%|▍                         | 350/21152 [01:18<1:24:00,  4.13it/s]


Ingestion:   2%|▍                         | 400/21152 [01:27<1:17:34,  4.46it/s]


Ingestion:   2%|▌                         | 450/21152 [02:04<2:12:56,  2.60it/s]


Ingestion:   2%|▌                         | 500/21152 [02:21<2:07:12,  2.71it/s]


Ingestion:   3%|▋                         | 550/21152 [02:30<1:47:24,  3.20it/s]


I

Batch-Fehler: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The service is currently unavailable.', 'status': 'UNAVAILABLE'}}





Ingestion:  76%|███████████████████▊      | 16100/21152 [55:08<22:44,  3.70it/s]


Ingestion:  76%|███████████████████▊      | 16150/21152 [55:21<22:09,  3.76it/s]


Ingestion:  77%|███████████████████▉      | 16200/21152 [55:28<18:50,  4.38it/s]


Ingestion:  77%|███████████████████▉      | 16250/21152 [55:35<16:38,  4.91it/s]


Ingestion:  77%|████████████████████      | 16300/21152 [55:43<15:11,  5.32it/s]


Ingestion:  77%|████████████████████      | 16350/21152 [55:55<16:08,  4.96it/s]


Ingestion:  78%|████████████████████▏     | 16400/21152 [56:04<15:33,  5.09it/s]


Ingestion:  78%|████████████████████▏     | 16450/21152 [56:11<14:12,  5.51it/s]


Ingestion:  78%|████████████████████▎     | 16500/21152 [56:20<14:00,  5.54it/s]


Ingestion:  78%|████████████████████▎     | 16550/21152 [56:26<12:21,  6.21it/s]


Ingestion:  78%|████████████████████▍     | 16600/21152 [56:38<14:07,  5.37it/s]


Ingestion:  79%|████████████████████▍     | 16650/21152 [56:47<13:56,  5.38it/s]


I